# Task 3: Hybrid + rerank

Two steps on top of the three retrievers from notebooks 1 and 2:

1. **Hybrid**: combine the BM25 ranking with the dense rankings (rank fusion and score interpolation are both tried, val decides).
2. **Rerank**: take the top-k of the best first-stage system and re-order it with a cross-encoder, which reads claim and abstract together instead of comparing two separate vectors.

Needs the zip from notebook 2. GPU runtime (T4), about 5-8 min.

In [1]:
# upload NLP_Inter_IIT.zip (the one notebook 2 downloaded in its last cell) and unpack it
# on colab this opens the file picker. If the folder is already here (rerun, or running locally) nothing happens.
import zipfile
from pathlib import Path

def find_root():
    # folder that contains data/, wherever the notebook was opened from
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "NLP_Inter_IIT"]:
        if (p / "data" / "corpus.jsonl").exists():
            return p

if find_root() is None:
    zips = list(Path.cwd().glob("*.zip"))        # zip already dragged into the colab file panel
    if not zips:
        from google.colab import files
        zips = [Path(name) for name in files.upload()]
    newest = max(zips, key=lambda z: z.stat().st_mtime)
    zipfile.ZipFile(newest).extractall(".")
    print("unpacked", newest.name)

ROOT = find_root()
assert ROOT is not None, "data/corpus.jsonl not found, upload NLP_Inter_IIT.zip"
assert (ROOT / "work" / "scores_general_val.npy").exists(), "this zip has no dense scores: upload the zip that notebook 2 downloaded in its last cell"
print("project folder:", ROOT)
print(sorted(p.name for p in (ROOT / "work").iterdir()))

Saving NLP_Inter_IIT_3.zip to NLP_Inter_IIT_3.zip
unpacked NLP_Inter_IIT_3.zip
project folder: /content/NLP_Inter_IIT
['results_log.json', 'scores_bm25_eval.npy', 'scores_bm25_train.npy', 'scores_bm25_val.npy', 'scores_general_eval.npy', 'scores_general_train.npy', 'scores_general_val.npy', 'scores_scientific_eval.npy', 'scores_scientific_train.npy', 'scores_scientific_val.npy']


In [2]:
%pip install -q transformers

In [3]:
import json, math, sys, time, itertools

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DATA, PRED, WORK = ROOT / "data", ROOT / "predictions", ROOT / "work"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "none (cpu)")

cuda | Tesla T4


## Data and helpers (same as before)

In [4]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

corpus = sorted(read_jsonl(DATA / "corpus.jsonl"), key=lambda d: d["doc_id"])
doc_ids = np.array([d["doc_id"] for d in corpus])
claims = {s: sorted(read_jsonl(DATA / f"{s}_claims.jsonl"), key=lambda c: c["id"]) for s in ["train", "val", "eval"]}

TOP_K = 100

def gold_docs(split):
    return {c["id"]: {e["doc_id"] for e in c["evidence"]} for c in claims[split]}

def make_run(scores, split, k=TOP_K):
    order = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    run = {}
    for i, c in enumerate(claims[split]):
        run[c["id"]] = [(int(doc_ids[j]), float(scores[i, j])) for j in order[i]]
    return run

def claim_metrics(run, gold):
    rows = {}
    for cid, g in gold.items():
        if len(g) == 0:
            continue
        ranked = [d for d, _ in run[cid]]
        dcg = sum(1 / math.log2(r + 2) for r, d in enumerate(ranked[:10]) if d in g)
        idcg = sum(1 / math.log2(r + 2) for r in range(min(len(g), 10)))
        rr = 0.0
        for r, d in enumerate(ranked):
            if d in g:
                rr = 1 / (r + 1)
                break
        rows[cid] = {"ndcg10": dcg / idcg, "recall100": len(g & set(ranked)) / len(g), "mrr": rr}
    return pd.DataFrame.from_dict(rows, orient="index")

def evaluate(run, gold):
    return claim_metrics(run, gold).mean().round(4).to_dict()

def write_trec(path, run, name):
    with open(path, "w") as f:
        for cid in sorted(run):
            for rank, (did, score) in enumerate(run[cid], start=1):
                f.write(f"{cid} Q0 {did} {rank} {score:.6f} {name}\n")

results_log = json.load(open(WORK / "results_log.json"))
print("general   :", results_log["task2"]["general_model"])
print("scientific:", results_log["task2"]["scientific_model"])

general   : BAAI/bge-large-en-v1.5
scientific: ncbi/MedCPT-Query-Encoder + ncbi/MedCPT-Article-Encoder


In [5]:
# score matrices from notebooks 1 and 2: scores[system][split] has shape (n_claims, 5183)
SYSTEMS = ["bm25", "general", "scientific"]
scores = {name: {s: np.load(WORK / f"scores_{name}_{s}.npy") for s in ["train", "val", "eval"]} for name in SYSTEMS}

def val_ndcg(mat):
    return evaluate(make_run(mat, "val"), gold_docs("val"))["ndcg10"]

single = pd.DataFrame({name: evaluate(make_run(scores[name]["val"], "val"), gold_docs("val")) for name in SYSTEMS}).T
single

,ndcg10,recall100,mrr
bm25,0.8735,0.9811,0.8546
general,0.9328,1.0000,0.9191
scientific,0.8509,1.0000,0.8127


## Hybrid

Two ways to combine rankings:

**Reciprocal rank fusion (RRF)** only uses the rank of a document in each system:

$$\text{RRF}(d)=\sum_{s} \frac{w_s}{k+\text{rank}_s(d)}$$

No score normalisation needed, since BM25 scores and embedding similarities live on different scales. With $w_s=1$ and $k=60$ this is the standard parameter-free version. A weight on the dense side lets the stronger system count more.

**Score interpolation** uses the actual scores: each system's scores are z-normalised per claim and then mixed with weights that sum to 1.

Everything is evaluated on val and the best val nDCG@10 is kept. Train is shown as a second opinion (no system has been trained on these claims yet).

In [6]:
def to_ranks(mat):
    # rank of every abstract for every claim (1 = best)
    order = np.argsort(-mat, axis=1, kind="stable")
    ranks = np.empty_like(order)
    ranks[np.arange(mat.shape[0])[:, None], order] = np.arange(1, mat.shape[1] + 1)
    return ranks

ranks = {name: {s: to_ranks(scores[name][s]) for s in ["train", "val", "eval"]} for name in SYSTEMS}

def zscore(mat):
    return (mat - mat.mean(axis=1, keepdims=True)) / (mat.std(axis=1, keepdims=True) + 1e-9)

zscores = {name: {s: zscore(scores[name][s].astype(np.float64)) for s in ["train", "val", "eval"]} for name in SYSTEMS}

def rrf(split, weights, k=60):
    # weights: {system: weight}
    return sum(w / (k + ranks[name][split]) for name, w in weights.items())

def interpolate(split, weights):
    return sum(w * zscores[name][split] for name, w in weights.items())

In [7]:
# candidate hybrids: (label, function, weights, extra args). The plain parameter-free ones come first,
# so on a tie the simplest method is kept.
candidates = [
    ("RRF bm25+general (k=60)", rrf, {"bm25": 1, "general": 1}, {"k": 60}),
    ("RRF bm25+general+scientific (k=60)", rrf, {"bm25": 1, "general": 1, "scientific": 1}, {"k": 60}),
]
for w, k in itertools.product([2, 3, 5], [20, 60]):
    candidates.append((f"weighted RRF bm25+general (general x{w}, k={k})", rrf, {"bm25": 1, "general": w}, {"k": k}))
for w in [2, 3]:
    candidates.append((f"weighted RRF all three (general x{w}, k=60)", rrf, {"bm25": 1, "general": w, "scientific": 1}, {"k": 60}))
for a in [0.5, 0.6, 0.7, 0.8, 0.9]:
    candidates.append((f"z-score interpolation {a:.1f} general + {1 - a:.1f} bm25", interpolate,
                       {"general": a, "bm25": round(1 - a, 2)}, {}))
for a, b in [(0.5, 0.25), (0.6, 0.2), (0.7, 0.15)]:
    candidates.append((f"z-score interpolation {a} general + {b} bm25 + {round(1 - a - b, 2)} scientific", interpolate,
                       {"general": a, "bm25": b, "scientific": round(1 - a - b, 2)}, {}))

rows = []
for i, (label, fn, weights, kwargs) in enumerate(candidates):
    val = evaluate(make_run(fn("val", weights, **kwargs), "val"), gold_docs("val"))
    train = evaluate(make_run(fn("train", weights, **kwargs), "train"), gold_docs("train"))
    rows.append({"method": label, "val_ndcg10": val["ndcg10"], "val_recall100": val["recall100"],
                 "val_mrr": val["mrr"], "train_ndcg10": train["ndcg10"], "idx": i})

hybrid_table = pd.DataFrame(rows).sort_values("val_ndcg10", ascending=False, kind="stable").reset_index(drop=True)
print("best single system on val:", single["ndcg10"].idxmax(), single["ndcg10"].max())
hybrid_table.drop(columns="idx")

best single system on val: general 0.9328


,method,val_ndcg10,val_recall100,val_mrr,train_ndcg10
0,z-score interpolation 0.7 general + 0.15 bm25 ...,0.9352,1.0000,0.9197,0.9277
1,z-score interpolation 0.9 general + 0.1 bm25,0.9342,1.0000,0.9182,0.9283
2,"weighted RRF bm25+general (general x5, k=20)",0.9282,1.0000,0.9120,0.9228
3,"weighted RRF all three (general x2, k=60)",0.9269,1.0000,0.9107,0.9220
4,z-score interpolation 0.6 general + 0.2 bm25 +...,0.9245,1.0000,0.9077,0.9237
5,RRF bm25+general+scientific (k=60),0.9238,1.0000,0.9067,0.9212
6,"weighted RRF bm25+general (general x5, k=60)",0.9234,1.0000,0.9060,0.9222
7,"weighted RRF bm25+general (general x3, k=20)",0.9231,1.0000,0.9064,0.9205
8,"weighted RRF all three (general x3, k=60)",0.9228,1.0000,0.9023,0.9230
9,z-score interpolation 0.8 general + 0.2 bm25,0.9207,1.0000,0.8987,0.9232


In [8]:
# keep the best hybrid, save its score matrices and write the eval run file
best_row = hybrid_table.iloc[0]
hybrid_method, fn, weights, kwargs = candidates[int(best_row["idx"])]
print("selected hybrid:", hybrid_method)

scores["hybrid"] = {}
for split in ["train", "val", "eval"]:
    scores["hybrid"][split] = fn(split, weights, **kwargs)
    np.save(WORK / f"scores_hybrid_{split}.npy", scores["hybrid"][split].astype(np.float32))

write_trec(PRED / "t3_hybrid.trec", make_run(scores["hybrid"]["eval"], "eval"), "hybrid")

first_stage = pd.DataFrame({name: evaluate(make_run(scores[name]["val"], "val"), gold_docs("val"))
                            for name in ["bm25", "general", "scientific", "hybrid"]}).T
first_stage

selected hybrid: z-score interpolation 0.7 general + 0.15 bm25 + 0.15 scientific


,ndcg10,recall100,mrr
bm25,0.8735,0.9811,0.8546
general,0.9328,1.0000,0.9191
scientific,0.8509,1.0000,0.8127
hybrid,0.9352,1.0000,0.9197


## Rerank

A bi-encoder embeds claim and abstract separately, so the two texts never "see" each other. A cross-encoder gets `[claim] [SEP] [abstract]` as one input and outputs a single relevance score, which is more accurate but far too slow to run on all 5183 abstracts. So it is only used on the top-k of the best first-stage system.

Format rule from the problem statement: the k reranked abstracts go to ranks 1..k in cross-encoder order, ranks k+1..100 keep the first-stage order.

Which cross-encoder and which k are both picked on val. All candidates are under 400M parameters and used zero-shot.

In [9]:
# best first-stage system = highest val nDCG@10 among the four
best_first = first_stage["ndcg10"].idxmax()
print("reranking the top-k of:", best_first, "| val nDCG@10 =", first_stage.loc[best_first, "ndcg10"])

first_runs = {split: make_run(scores[best_first][split], split) for split in ["val", "eval"]}

RERANKERS = {
    "bge-reranker-base": "BAAI/bge-reranker-base",                            # 278M, general
    "MedCPT-Cross-Encoder": "ncbi/MedCPT-Cross-Encoder",                      # 109M, biomedical
    "ms-marco-MiniLM-L-12-v2": "cross-encoder/ms-marco-MiniLM-L-12-v2",       # 33M, general, fast
}
RERANK_DEPTH = 100                       # val is scored down to this depth once, then any k <= depth is free
K_GRID = [k for k in [3, 5, 10, 20, 30, 50, 100] if k <= RERANK_DEPTH]

docs_by_id = {d["doc_id"]: d["title"] + " " + " ".join(d["abstract"]) for d in corpus}
claim_text = {split: {c["id"]: c["claim"] for c in claims[split]} for split in ["val", "eval"]}

@torch.no_grad()
def cross_scores(tok, model, queries, docs, batch_size=64):
    # one relevance score per (claim, abstract) pair. Only the abstract is truncated if the pair is too long.
    out = []
    for i in range(0, len(queries), batch_size):
        batch = tok(queries[i:i + batch_size], docs[i:i + batch_size], padding=True, truncation="only_second",
                    max_length=512, return_tensors="pt").to(device)
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            logits = model(**batch).logits
        out.append(logits.float()[:, -1].cpu().numpy())      # all three models have a single output logit
    return np.concatenate(out)

def score_top(tok, model, split, depth):
    # cross-encoder scores for the top `depth` first-stage abstracts of every claim: {claim_id: array of length depth}
    queries, docs, owner = [], [], []
    for cid, ranked in first_runs[split].items():
        for did, _ in ranked[:depth]:
            queries.append(claim_text[split][cid]); docs.append(docs_by_id[did]); owner.append(cid)
    flat = cross_scores(tok, model, queries, docs)
    return {cid: flat[i * depth:(i + 1) * depth] for i, cid in enumerate(first_runs[split])}

def rerank(first_run, ce, k):
    # top-k in cross-encoder order, then the remaining abstracts in their first-stage order
    out = {}
    for cid, ranked in first_run.items():
        head = sorted(zip([d for d, _ in ranked[:k]], ce[cid][:k].tolist()), key=lambda x: -x[1])
        floor = min(s for _, s in head)
        tail = [(d, floor - 1 - j) for j, (d, _) in enumerate(ranked[k:])]     # scores must keep decreasing
        out[cid] = head + tail
    return out

reranking the top-k of: hybrid | val nDCG@10 = 0.9352


In [10]:
# score val with every cross-encoder, then try every k
ce_val, rows = {}, []
for short, name in RERANKERS.items():
    try:
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModelForSequenceClassification.from_pretrained(name).to(device).eval()
    except Exception as e:
        print(f"skipping {short}: {type(e).__name__}: {e}")
        continue
    n_par = sum(p.numel() for p in model.parameters())
    assert n_par <= 400e6
    t0 = time.perf_counter()
    ce_val[short] = score_top(tok, model, "val", RERANK_DEPTH)
    print(f"{short}: {n_par / 1e6:.0f}M params, val scored in {time.perf_counter() - t0:.0f}s")
    for k in K_GRID:
        m = evaluate(rerank(first_runs["val"], ce_val[short], k), gold_docs("val"))
        rows.append({"reranker": short, "k": k, **m})
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

rerank_table = pd.DataFrame(rows)
print(f"first stage ({best_first}) val nDCG@10 = {first_stage.loc[best_first, 'ndcg10']}")
rerank_table.pivot(index="reranker", columns="k", values="ndcg10")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ms-marco-MiniLM-L-12-v2: 33M params, val scored in 75s
first stage (hybrid) val nDCG@10 = 0.9352


k,3,5,10,20,30,50,100
reranker,,,,,,,
MedCPT-Cross-Encoder,0.9259,0.9177,0.9149,0.9094,0.9125,0.9125,0.9125
bge-reranker-base,0.9090,0.8857,0.8725,0.8583,0.8522,0.8496,0.8473
ms-marco-MiniLM-L-12-v2,0.9087,0.9074,0.8869,0.8666,0.8637,0.8637,0.8633


In [11]:
# best (cross-encoder, k) on val. On a tie the smaller k wins, since it is cheaper.
best = rerank_table.sort_values(["ndcg10", "k"], ascending=[False, True], kind="stable").iloc[0]
reranker_short, rerank_k = best["reranker"], int(best["k"])
reranker_name = RERANKERS[reranker_short]
print(f"selected: {reranker_short} with k = {rerank_k} | val nDCG@10 = {best['ndcg10']}")

tok = AutoTokenizer.from_pretrained(reranker_name)
model = AutoModelForSequenceClassification.from_pretrained(reranker_name).to(device).eval()

# eval claims only need the top-k scored
ce_eval = score_top(tok, model, "eval", rerank_k)
rerank_runs = {"val": rerank(first_runs["val"], ce_val[reranker_short], rerank_k),
               "eval": rerank(first_runs["eval"], ce_eval, rerank_k)}

# latency: one claim at a time, k pairs per claim
def sync():
    if device == "cuda":
        torch.cuda.synchronize()

times = []
for n, (cid, ranked) in enumerate(first_runs["val"].items()):
    docs = [docs_by_id[d] for d, _ in ranked[:rerank_k]]
    sync(); t0 = time.perf_counter()
    cross_scores(tok, model, [claim_text["val"][cid]] * rerank_k, docs, batch_size=rerank_k)
    sync()
    if n >= 5:                                   # first few calls are warm-up
        times.append((time.perf_counter() - t0) * 1000)
rerank_latency = round(float(np.mean(times)), 2)
print(f"rerank latency: {rerank_latency} ms per claim for {rerank_k} pairs")

del model
if device == "cuda":
    torch.cuda.empty_cache()

selected: MedCPT-Cross-Encoder with k = 3 | val nDCG@10 = 0.9259


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

rerank latency: 24.55 ms per claim for 3 pairs


In [12]:
# run file + format check
write_trec(PRED / "t3_rerank.trec", rerank_runs["eval"], "rerank")

sys.path.append(str(ROOT))
from check_format import check_trec

all_docs = set(doc_ids.tolist())
eval_ids = {c["id"] for c in claims["eval"]}
check_trec(PRED / "t3_hybrid.trec", eval_ids, all_docs)
check_trec(PRED / "t3_rerank.trec", eval_ids, all_docs)
print("t3_hybrid.trec and t3_rerank.trec pass check_format")

# the reranked rankings are kept for the verification step (task 5)
for split in ["val", "eval"]:
    json.dump({str(cid): rows for cid, rows in rerank_runs[split].items()}, open(WORK / f"run_rerank_{split}.json", "w"))

t3_hybrid.trec and t3_rerank.trec pass check_format


## Summary (this is what report Q2 asks about)

In [13]:
val_rerank = evaluate(rerank_runs["val"], gold_docs("val"))
best_single = single["ndcg10"].idxmax()

summary = pd.DataFrame({
    "bm25": first_stage.loc["bm25"],
    "general": first_stage.loc["general"],
    "scientific": first_stage.loc["scientific"],
    "hybrid": first_stage.loc["hybrid"],
    "rerank": pd.Series(val_rerank),
}).T
print(summary)

gain_hybrid = first_stage.loc["hybrid", "ndcg10"] - single["ndcg10"].max()
gain_rerank = val_rerank["ndcg10"] - first_stage.loc[best_first, "ndcg10"]
dense_latency = results_log["task2"]["stats"]["general"]["query_latency_ms"]
print(f"\nhybrid vs best single system ({best_single}): {gain_hybrid:+.4f} nDCG@10")
print(f"rerank vs its first stage ({best_first}): {gain_rerank:+.4f} nDCG@10")
print(f"latency: dense query = {dense_latency} ms, reranking the top {rerank_k} adds {rerank_latency} ms per claim")

            ndcg10  recall100     mrr
bm25        0.8735     0.9811  0.8546
general     0.9328     1.0000  0.9191
scientific  0.8509     1.0000  0.8127
hybrid      0.9352     1.0000  0.9197
rerank      0.9259     1.0000  0.9072

hybrid vs best single system (general): +0.0024 nDCG@10
rerank vs its first stage (hybrid): -0.0093 nDCG@10
latency: dense query = 46.41 ms, reranking the top 3 adds 24.55 ms per claim


In [14]:
results_log["task3"] = {
    "hybrid_method": hybrid_method,
    "reranker_model": reranker_name,
    "rerank_top_k": rerank_k,
    "val_ndcg10_hybrid": float(first_stage.loc["hybrid", "ndcg10"]),
    "val_ndcg10_rerank": float(val_rerank["ndcg10"]),
    "rerank_query_latency_ms": rerank_latency,
    "reranked_system": best_first,
    "gain_hybrid_vs_best_single": round(float(gain_hybrid), 4),
    "gain_rerank_vs_first_stage": round(float(gain_rerank), 4),
    "val": json.loads(summary.to_json(orient="index")),
    "hybrid_candidates": json.loads(hybrid_table.drop(columns="idx").to_json(orient="records")),
    "rerank_candidates": json.loads(rerank_table.to_json(orient="records")),
}
json.dump(results_log, open(WORK / "results_log.json", "w"), indent=2)
{k: results_log["task3"][k] for k in ["hybrid_method", "reranker_model", "rerank_top_k", "val_ndcg10_hybrid", "val_ndcg10_rerank", "rerank_query_latency_ms"]}

{'hybrid_method': 'z-score interpolation 0.7 general + 0.15 bm25 + 0.15 scientific',
 'reranker_model': 'ncbi/MedCPT-Cross-Encoder',
 'rerank_top_k': 3,
 'val_ndcg10_hybrid': 0.9352,
 'val_ndcg10_rerank': 0.9259,
 'rerank_query_latency_ms': 24.55}

## Save for the next notebook

Packs data, predictions and work into `NLP_Inter_IIT.zip` and downloads it (colab only). Upload this zip in the first cell of the next notebook.

In [15]:
if "google.colab" in sys.modules:
    zip_path = Path.cwd() / "NLP_Inter_IIT_4.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for item in ["data", "predictions", "work", "check_format.py", "results_template.json"]:
            p = ROOT / item
            if not p.exists():
                continue
            for f in ([p] if p.is_file() else sorted(p.rglob("*"))):
                if f.is_file():
                    z.write(f, Path("NLP_Inter_IIT") / f.relative_to(ROOT))
    print(zip_path.name, f"{zip_path.stat().st_size / 1e6:.0f} MB")
    from google.colab import files
    files.download(str(zip_path))

NLP_Inter_IIT_4.zip 84 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>